# Loan Dataset

In this notebook, the loan dataset will be read in, prepared, analyzed and predicted. 

## 1.1 Notebook Preperation

To prepare the notebook to conduct these tasks, first we will import the necessary libraries, configure logging and read in the required dataset.

In [1]:
# Import libraries for notebook cell execution
import pandas as pd
import numpy as np 
import plotly.express as px
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MaxAbsScaler, StandardScaler, OrdinalEncoder, FunctionTransformer, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTENC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, f1_score
import logging
import shap
import sys

In [2]:
# Create logger instance
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)


In [3]:
df_loan = pd.read_csv("../data/loan-10k.lrn.csv")
target_col = 'loan_grade'
X = df_loan.drop(columns=["grade"])
y = df_loan[["grade"]] 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=12347303,
    stratify=y
)
loan_train_df = pd.concat([X_train, y_train], axis=1)
loan_test_df = pd.concat([X_test, y_test], axis=1)
loan_test_final_df = pd.read_csv("../data/loan-10k.tes.csv")

## 1.2 Data Analysis

In [4]:
# Display dataset information:
loan_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8000 entries, 8621 to 5982
Data columns (total 92 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          8000 non-null   int64  
 1   loan_amnt                   8000 non-null   float64
 2   funded_amnt                 8000 non-null   float64
 3   funded_amnt_inv             8000 non-null   float64
 4   term                        8000 non-null   object 
 5   int_rate                    8000 non-null   float64
 6   installment                 8000 non-null   float64
 7   emp_length                  8000 non-null   object 
 8   home_ownership              8000 non-null   object 
 9   annual_inc                  8000 non-null   float64
 10  verification_status         8000 non-null   object 
 11  loan_status                 8000 non-null   object 
 12  pymnt_plan                  8000 non-null   object 
 13  purpose                     8000 no

In [5]:
# Display first rows of the dataframe:
display(loan_train_df.head())

,ID,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,emp_length,home_ownership,annual_inc,...,debt_settlement_flag,issue_d_month,issue_d_year,earliest_cr_line_month,earliest_cr_line_year,last_pymnt_d_month,last_pymnt_d_year,last_credit_pull_d_month,last_credit_pull_d_year,grade
8621,96372,40000.0,40000.0,40000.0,60 months,17.97,1015.09,6 years,MORTGAGE,190000.0,...,N,10,2018,9,2001,2,2019,2,2019,D
4859,26751,15000.0,15000.0,15000.0,36 months,11.49,494.57,10+ years,MORTGAGE,70000.0,...,N,7,2016,7,2005,8,2016,10,2018,B
1943,88879,20000.0,20000.0,20000.0,60 months,13.99,465.27,10+ years,MORTGAGE,135000.0,...,N,9,2015,9,1997,3,2016,0,2017,C
4223,47225,9450.0,9450.0,9450.0,36 months,23.43,367.93,10+ years,OWN,31720.0,...,N,7,2014,0,2000,8,2015,1,2017,F
9919,88712,25700.0,25700.0,25700.0,36 months,16.02,903.79,10+ years,RENT,150000.0,...,N,5,2017,7,2005,2,2019,2,2019,C


In [6]:
# Datset Dimensions
logger.info(f"Rows: {loan_train_df.shape[0]} Columns: {loan_train_df.shape[1]}")

23:35:19 [INFO] __main__ - Rows: 8000 Columns: 92


In [7]:
# Retrieve missing values
logger.info("Missing Values Per Column:")
display(loan_train_df.isnull().sum())
logger.info(f"Total missing values: {loan_train_df.isnull().sum().sum()}")

23:35:19 [INFO] __main__ - Missing Values Per Column:


ID                          0
loan_amnt                   0
funded_amnt                 0
funded_amnt_inv             0
term                        0
                           ..
last_pymnt_d_month          0
last_pymnt_d_year           0
last_credit_pull_d_month    0
last_credit_pull_d_year     0
grade                       0
Length: 92, dtype: int64

23:35:19 [INFO] __main__ - Total missing values: 0


There are no missing values that need to be taken care of.

In [8]:
# Value distribution of target class: grade
logger.info("Target Variable: Grade")
loan_train_class_df = loan_train_df["grade"].value_counts()
logger.info(loan_train_class_df.head(20))
logger.info("Count of distinct Review Authors: " + str(loan_train_class_df.size))

23:35:19 [INFO] __main__ - Target Variable: Grade
23:35:19 [INFO] __main__ - grade
C    2391
B    2305
A    1457
D    1159
E     497
F     145
G      46
Name: count, dtype: int64
23:35:19 [INFO] __main__ - Count of distinct Review Authors: 7


In [9]:
# Plot distribution of target class: grade
fig_grade = px.histogram(
    loan_train_df,
    x="grade",
    color="grade",
    category_orders={"grade": sorted(df_loan["grade"].unique())},
    text_auto=True
)
fig_grade.show()

When looking at the plot, we can see at the first glance the large imbalance of the target class grade. The grades C,B are the most common grades, while grade E,F and G are hugely underrepresent in the dataset. For even better clarity, we output the percentage distribution. 

In [10]:
# Numeric summary
grade_counts = loan_train_df["grade"].value_counts().sort_index()
grade_probability = loan_train_df["grade"].value_counts(normalize=True).sort_index()

balance_df = pd.DataFrame({
    "count": grade_counts,
    "proportion": grade_probability
})

logger.info(balance_df.sort_values(by="proportion", ascending=False))

23:35:20 [INFO] __main__ -        count  proportion
grade                   
C       2391    0.298875
B       2305    0.288125
A       1457    0.182125
D       1159    0.144875
E        497    0.062125
F        145    0.018125
G         46    0.005750


In [11]:
# Check for non-numeric features 
non_numeric_cols = loan_train_df.select_dtypes(exclude=['number']).columns
logging.info(non_numeric_cols)
num_non_numeric = len(non_numeric_cols)
logging.info(f"Number of non-numeric columns: {num_non_numeric}")

23:35:20 [INFO] root - Index(['term', 'emp_length', 'home_ownership', 'verification_status',
       'loan_status', 'pymnt_plan', 'purpose', 'addr_state',
       'initial_list_status', 'application_type', 'hardship_flag',
       'disbursement_method', 'debt_settlement_flag', 'grade'],
      dtype='object')
23:35:20 [INFO] root - Number of non-numeric columns: 14


There are 14 non-numeric columns that need potential encoding to numeric columns. 

For an accurate handling of outliers we first have to encode the non-numerical values in order to perform z-score evaluation. Therefore it will be moved to data preprocessing. 

## 1.3 Data Preprocessing 

No missing values have to be handled, therefore this step will be skipped. 
Now we will handle to encode the non-numeric values to be better processable by the used models. 
We first have to identify if the data is ordinal or nominal categorical values in order to know to encode it. 
Therefore, we will investigate the 14 non-numerical columns of the dataset. 

In [12]:
# Display df of non-numeric values
display(loan_train_df[non_numeric_cols])
# Display unique values 
for column in non_numeric_cols:
    logger.info(f"Unique values for columns: {column} {loan_train_df[column].unique()}")
    logger.info(loan_train_df[column].value_counts(normalize=True))

,term,emp_length,home_ownership,verification_status,loan_status,pymnt_plan,purpose,addr_state,initial_list_status,application_type,hardship_flag,disbursement_method,debt_settlement_flag,grade
8621,60 months,6 years,MORTGAGE,Source Verified,Current,n,debt_consolidation,MD,w,Joint App,N,Cash,N,D
4859,36 months,10+ years,MORTGAGE,Verified,Fully Paid,n,debt_consolidation,OK,w,Individual,N,Cash,N,B
1943,60 months,10+ years,MORTGAGE,Source Verified,Fully Paid,n,car,TX,w,Individual,N,Cash,N,C
4223,36 months,10+ years,OWN,Source Verified,Charged Off,n,debt_consolidation,VT,f,Individual,N,Cash,N,F
9919,36 months,10+ years,RENT,Verified,Current,n,debt_consolidation,MN,w,Individual,N,Cash,N,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5575,60 months,10+ years,MORTGAGE,Verified,Current,n,credit_card,TX,w,Joint App,N,Cash,N,C
6661,36 months,< 1 year,RENT,Verified,Charged Off,n,debt_consolidation,MN,f,Individual,N,Cash,N,D
8881,60 months,10+ years,MORTGAGE,Source Verified,Fully Paid,n,debt_consolidation,MN,f,Individual,N,Cash,N,C
95,36 months,9 years,RENT,Not Verified,Current,n,home_improvement,SC,w,Individual,N,Cash,N,B


23:35:20 [INFO] __main__ - Unique values for columns: term [' 60 months' ' 36 months']
23:35:20 [INFO] __main__ - term
36 months    0.69375
60 months    0.30625
Name: proportion, dtype: float64
23:35:20 [INFO] __main__ - Unique values for columns: emp_length ['6 years' '10+ years' '8 years' '3 years' '4 years' '1 year' '2 years'
 '5 years' '9 years' '< 1 year' '7 years']
23:35:20 [INFO] __main__ - emp_length
10+ years    0.354625
2 years      0.100375
3 years      0.085125
< 1 year     0.084500
5 years      0.068250
1 year       0.067625
4 years      0.063875
6 years      0.052375
7 years      0.047375
9 years      0.038625
8 years      0.037250
Name: proportion, dtype: float64
23:35:20 [INFO] __main__ - Unique values for columns: home_ownership ['MORTGAGE' 'OWN' 'RENT' 'OTHER' 'ANY']
23:35:20 [INFO] __main__ - home_ownership
MORTGAGE    0.504125
RENT        0.384000
OWN         0.111625
OTHER       0.000125
ANY         0.000125
Name: proportion, dtype: float64
23:35:20 [INFO] __main__

From these unique values we can gather the type of non-numerical values and therefore the encoding techniques:
term -> direct cast to month duration. As we also have svm which responds 
emp year: cast to years, <1 year will be cast to 0.5, 10+ to 10
home ownership will become one hot 
verification_status will become ordinal as it can be ranked
loan_status is also ranked so therefore we can rank the statuses from good to worse 
payment plan will become binary 0 1
purpose categorical also become one hot 
addr_state non categorical but too many adresses 
initial_list_status binary 0 1
application type binary 0 1
hardship_flag binary 0 1
disbursement_method binary 0 1 
debt_settlement_flag binary 0 1
grade ordinal encoding as grades have ranking 

In [13]:
loan_train_encoded_df = loan_train_df.copy()

In [14]:
# Encode binary columns
binary_columns = {"pymnt_plan", "initial_list_status", "application_type", "hardship_flag", "disbursement_method", "debt_settlement_flag"}
le = LabelEncoder()
for column in binary_columns:
    loan_train_encoded_df[column] = le.fit_transform(loan_train_df[column])
    logger.info(f"Unique label values of col: {column}: {loan_train_encoded_df[column].unique()}")
# Verify binary columns encoding 
display(loan_train_encoded_df[list(binary_columns)])

23:35:20 [INFO] __main__ - Unique label values of col: pymnt_plan: [0 1]
23:35:20 [INFO] __main__ - Unique label values of col: application_type: [1 0]
23:35:20 [INFO] __main__ - Unique label values of col: disbursement_method: [0 1]
23:35:20 [INFO] __main__ - Unique label values of col: debt_settlement_flag: [0 1]
23:35:20 [INFO] __main__ - Unique label values of col: initial_list_status: [1 0]
23:35:20 [INFO] __main__ - Unique label values of col: hardship_flag: [0 1]


,pymnt_plan,application_type,disbursement_method,debt_settlement_flag,initial_list_status,hardship_flag
8621,0,1,0,0,1,0
4859,0,0,0,0,1,0
1943,0,0,0,0,1,0
4223,0,0,0,0,0,0
9919,0,0,0,0,1,0
...,...,...,...,...,...,...
5575,0,1,0,0,1,0
6661,0,0,0,0,0,0
8881,0,0,0,0,0,0
95,0,0,0,0,1,0


In [15]:
# Encode ordinal values to ranked 
ordinal_mapping = {
    "grade" : {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5, "G": 6},
    "verification_status" : {"Verified": 0, "Source Verified": 1, "Not Verified": 2},
    "loan_status" : {"Fully Paid": 0, "Current": 1, "In Grace Period": 2, "Late (16-30 days)":3, "Late (31-120 days)": 4, "Charged Off": 5 },
}
loan_train_encoded_df = loan_train_encoded_df.replace(ordinal_mapping)
# Verify mapping
unique_per_column = {col: loan_train_encoded_df[col].unique() for col in ordinal_mapping.keys()}
logging.info(unique_per_column)

/var/folders/ld/yxrhvq2x3zzcf_rdhkny7drw0000gp/T/ipykernel_54901/1254616005.py:7: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

23:35:20 [INFO] root - {'grade': array([3, 1, 2, 5, 0, 6, 4]), 'verification_status': array([1, 0, 2]), 'loan_status': array([1, 0, 5, 4, 2, 3])}


In [16]:
# Encode direct numerical mapping 
# For the term column just extract the number 
loan_train_encoded_df['term'] = loan_train_df['term'].str.extract(r'(\d+)').astype(int)
logging.info(loan_train_encoded_df['term'].unique())

# years helper function 
def convert_years(x):
    x = x.strip()
    if '< 1' in x:
        return 0.5
    elif '10+' in x:
        return 10
    else:
        num = pd.to_numeric(''.join(filter(str.isdigit, x)))
        return num

loan_train_encoded_df['emp_length'] = loan_train_df['emp_length'].apply(convert_years)

23:35:20 [INFO] root - [60 36]


In [17]:
# One Hot Encoding
one_hot_columns = ["home_ownership", "purpose", "addr_state"]
encoded_cols = []
for column in one_hot_columns:
    enc = OneHotEncoder(drop=None, sparse_output=False)
    transformed = enc.fit_transform(loan_train_encoded_df[[column]])
    col_names = [f"{column}_{cat}" for cat in enc.categories_[0]]
    df_enc = pd.DataFrame(transformed, columns=col_names, index=loan_train_encoded_df.index)
    loan_train_encoded_df = pd.concat([loan_train_encoded_df.drop(columns=[column]), df_enc], axis=1)
    encoded_cols.extend(col_names)

print(loan_train_encoded_df[encoded_cols])

      home_ownership_ANY  home_ownership_MORTGAGE  home_ownership_OTHER  \
8621                 0.0                      1.0                   0.0   
4859                 0.0                      1.0                   0.0   
1943                 0.0                      1.0                   0.0   
4223                 0.0                      0.0                   0.0   
9919                 0.0                      0.0                   0.0   
...                  ...                      ...                   ...   
5575                 0.0                      1.0                   0.0   
6661                 0.0                      0.0                   0.0   
8881                 0.0                      1.0                   0.0   
95                   0.0                      0.0                   0.0   
5982                 0.0                      1.0                   0.0   

      home_ownership_OWN  home_ownership_RENT  purpose_car  \
8621                 0.0             

In [18]:
logger.info(loan_train_encoded_df.shape[1])

23:35:20 [INFO] __main__ - 157


As we do not need an ID as it doest carry any meaningful information we can omit this feature. 

In [19]:
def detect_outliers_iqr(df, threshold=7.5):
    outlier_summary = {}

    for col in df.select_dtypes(include='number').columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR

        mask_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        num_outliers = mask_outliers.sum()
        total = df[col].notna().sum()

        if num_outliers > 0:
            outlier_summary[col] = {
                "num_outliers": num_outliers,
                "num_total": total,
                "percentage": round(100 * num_outliers / total, 2)
            }

    return pd.DataFrame(outlier_summary).T.sort_values(by="percentage", ascending=False)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Optional: prevent wrapping
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

outlier_report = detect_outliers_iqr(loan_train_encoded_df)
print(outlier_report)

                            num_outliers  num_total  percentage
num_accts_ever_120_pd             1928.0     8000.0       24.10
purpose_credit_card               1827.0     8000.0       22.84
delinq_2yrs                       1516.0     8000.0       18.95
tot_coll_amt                      1260.0     8000.0       15.75
pub_rec                           1250.0     8000.0       15.62
addr_state_CA                     1088.0     8000.0       13.60
pub_rec_bankruptcies               923.0     8000.0       11.54
home_ownership_OWN                 893.0     8000.0       11.16
addr_state_TX                      666.0     8000.0        8.32
recoveries                         643.0     8000.0        8.04
collection_recovery_fee            622.0     8000.0        7.78
addr_state_FL                      600.0     8000.0        7.50
addr_state_NY                      593.0     8000.0        7.41
purpose_home_improvement           501.0     8000.0        6.26
purpose_other                      483.0

In [20]:
encoded_loan_df = loan_train_encoded_df.drop(columns=['ID'])

In [21]:
# Identify numeric columns for suitable scaling 
one_hot_cols = [col for col in loan_train_encoded_df.columns if loan_train_encoded_df[col].nunique() == 2]
print(one_hot_cols)
numeric_cols = [col for col in loan_train_encoded_df.columns if col not in one_hot_cols]
sparsity = (loan_train_encoded_df[numeric_cols] == 0).mean()
sparsity = sparsity.sort_values(ascending=False)
print(sparsity)

['term', 'pymnt_plan', 'initial_list_status', 'application_type', 'acc_now_delinq', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'hardship_flag', 'disbursement_method', 'debt_settlement_flag', 'home_ownership_ANY', 'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_car', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding', 'addr_state_AK', 'addr_state_AL', 'addr_state_AR', 'addr_state_AZ', 'addr_state_CA', 'addr_state_CO', 'addr_state_CT', 'addr_state_DC', 'addr_state_DE', 'addr_state_FL', 'addr_state_GA', 'addr_state_HI', 'addr_state_ID', 'addr_state_IL', 'addr_state_IN', 'addr_state_KS', 'addr_state_KY', 'addr_state_LA', 'addr_state_MA', 'addr_state_MD', 'addr_state_ME', 'addr_state_MI', 'addr_state_MN', 'addr_state_MO', 'add

In [22]:
def detect_skewed_columns(df, skew_threshold=20):
    """
    Detect numeric columns with skewness above a threshold.

    Parameters:
    - df: pd.DataFrame
    - skew_threshold: float, columns with skewness above this are considered skewed

    Returns:
    - skewed_cols: list of column names
    - skew_values: pd.Series of skew values for the skewed columns
    """
    numeric_cols = df.select_dtypes(include='number').columns
    skew_values = df[numeric_cols].skew()
    
    skewed_cols = skew_values[skew_values.abs() > skew_threshold].index.tolist()
    return skewed_cols, skew_values[skewed_cols]

detect_skewed_columns(loan_train_encoded_df)

(['pymnt_plan',
  'tot_coll_amt',
  'delinq_amnt',
  'num_tl_120dpd_2m',
  'num_tl_30dpd',
  'hardship_flag',
  'home_ownership_ANY',
  'home_ownership_OTHER',
  'purpose_renewable_energy',
  'purpose_wedding',
  'addr_state_AK',
  'addr_state_DC',
  'addr_state_ID',
  'addr_state_ND',
  'addr_state_SD',
  'addr_state_VT',
  'addr_state_WY'],
 pymnt_plan                  44.696193
 tot_coll_amt                45.627477
 delinq_amnt                 61.956767
 num_tl_120dpd_2m            39.969985
 num_tl_30dpd                20.450228
 hardship_flag               44.696193
 home_ownership_ANY          89.442719
 home_ownership_OTHER        89.442719
 purpose_renewable_energy    44.696193
 purpose_wedding             63.233691
 addr_state_AK               24.751080
 addr_state_DC               22.297723
 addr_state_ID               23.846248
 addr_state_ND               22.297723
 addr_state_SD               20.450228
 addr_state_VT               23.033327
 addr_state_WY               23

In [23]:
def auto_scale(df, sparse_threshold=0.9):
    """
    Automatically scale columns based on type and sparsity.
    
    Parameters:
    - df: pd.DataFrame
    - sparse_threshold: fraction of zeros above which a numeric column is considered sparse
    
    Returns:
    - df_scaled: pd.DataFrame with scaled numeric columns
    """
    df_scaled = df.copy()
    
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            zero_fraction = (df[col] == 0).mean()
            if zero_fraction >= sparse_threshold:
                # Sparse numeric -> MaxAbsScaler
                scaler = MaxAbsScaler()
            else:
                # Dense numeric -> StandardScaler
                scaler = StandardScaler()
            
            df_scaled[col] = scaler.fit_transform(df[[col]])
        else:
            # Non-numeric -> leave untouched
            continue
    
    return df_scaled



In [24]:
# resample target variable credit grade 
from imblearn.over_sampling import SMOTE
import pandas as pd

X = loan_train_encoded_df.drop(columns=["grade"])
y = loan_train_encoded_df["grade"]

smote = SMOTE(random_state=42, k_neighbors=2)
X_res, y_res = smote.fit_resample(X, y)

loan_train_encoded_df = pd.concat(
    [pd.DataFrame(X_res, columns=X.columns),
     pd.Series(y_res, name="grade")],
    axis=1
)

# Check new class distribution
print(loan_train_encoded_df["grade"].value_counts())


grade
3    2391
1    2391
2    2391
5    2391
0    2391
6    2391
4    2391
Name: count, dtype: int64


In [25]:
loan_train_encoded_df["grade"].value_counts()

grade
3    2391
1    2391
2    2391
5    2391
0    2391
6    2391
4    2391
Name: count, dtype: int64

In [26]:
# Plot distribution of target class: grade
fig_grade = px.histogram(
    loan_train_encoded_df,
    x="grade",
    color="grade",
    category_orders={"grade": sorted(loan_train_encoded_df["grade"].unique())},
    text_auto=True
)
fig_grade.show()

## 2 Model Training

Test Encoded & Balanced
Test Scaled
Test Normalised 
Test Outlier Removal
Test Feature selection
Test Hyperparameter tuning

First we apply the same transformation we did while preprocessing. 

In [27]:
from sklearn.preprocessing import FunctionTransformer

class NamedFunctionTransformer(FunctionTransformer):
    def __init__(self, func=None, feature_names_out=None, **kwargs):
        super().__init__(func=func, **kwargs)
        self._feature_names_out = feature_names_out

    def get_feature_names_out(self, input_features=None):
        return self._feature_names_out if self._feature_names_out else input_features

In [28]:
def extract_term(X):
    X = X.copy()
    X['term'] = X['term'].astype(str).str.extract(r'(\d+)').astype(float)
    return X

def convert_emp_length(X):
    def convert_years(x):
        if pd.isna(x): return np.nan
        x = str(x).strip()
        if '< 1' in x: return 0.5
        if '10+' in x: return 10.0
        d = ''.join(filter(str.isdigit, x))
        return float(d) if d else np.nan
    X = X.copy()
    X['emp_length'] = X['emp_length'].apply(convert_years)
    return X

one_hot_cols = ["home_ownership","purpose","addr_state"]
ordinal_cols = ["verification_status","loan_status"]
ordinal_categories = [
    ["Verified","Source Verified","Not Verified"],
    ["Fully Paid","Current","In Grace Period","Late (16-30 days)","Late (31-120 days)","Default", "Charged Off"]
]
binary_cols = ["pymnt_plan","initial_list_status","application_type","hardship_flag","disbursement_method","debt_settlement_flag"]

preprocessor = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), one_hot_cols),
    ("ordinal", OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
    ("binary", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), binary_cols),
    ("term", NamedFunctionTransformer(extract_term, feature_names_out=["term"]), ["term"]),
    ("emp", NamedFunctionTransformer(convert_emp_length, feature_names_out=["emp_length"]), ["emp_length"])
], remainder="passthrough", verbose_feature_names_out=False)

pipeline = Pipeline([("preprocess", preprocessor)])




X_train_prep = pipeline.fit_transform(X_train)
X_test_prep = pipeline.transform(X_test)

X_train_prep_df = pd.DataFrame(
    X_train_prep,
    columns=pipeline.get_feature_names_out(),
    index=X_train.index
)

X_test_prep_df = pd.DataFrame(
    X_test_prep,
    columns=pipeline.get_feature_names_out(),
    index=X_test.index
)

grade_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5, "G": 6}
y_train_encoded = y_train.squeeze().map(grade_mapping)
y_test_encoded = y_test.squeeze().map(grade_mapping)
smote = SMOTE(random_state=12347303)
X_resampled, y_resampled = smote.fit_resample(X_train_prep, y_train_encoded)



reverse_grade_mapping = {v: k for k, v in grade_mapping.items()}


### 2.1 XG Boost

First lets try this set on only encoded data.

In [29]:
# for efficient xgboost use , convert to dmatrix 

xgb_initial = XGBClassifier(
    objective="multi:softmax",
    num_class=7,
    eval_metric="mlogloss",
    random_state=12347303
)

xgb_initial.fit(X_resampled, y_resampled)
y_pred_encoded = xgb_initial.predict(X_test_prep)
y_pred_labels = pd.Series(y_pred_encoded).map(reverse_grade_mapping)
y_test_series = y_test.squeeze()
y_test_labels = y_test_series.map(reverse_grade_mapping)

f1_macro = f1_score(y_test_encoded, y_pred_encoded, average="macro")
f1_weighted = f1_score(y_test_encoded, y_pred_encoded, average="weighted")

accuracy = accuracy_score(y_test_encoded, y_pred_encoded)
precision = precision_score(y_test_encoded, y_pred_encoded, average='macro')
recall = recall_score(y_test_encoded, y_pred_encoded, average='macro')
f1 = f1_score(y_test_encoded, y_pred_encoded, average='macro')


y_test_bin = label_binarize(y_test_encoded, classes=list(range(7)))
y_pred_proba = xgb_initial.predict_proba(X_test_prep)
roc_auc = roc_auc_score(y_test_bin, y_pred_proba, average='macro', multi_class='ovr')
pr_auc = average_precision_score(y_test_bin, y_pred_proba, average='macro')

# --- Print results ---
metrics_df = pd.DataFrame({
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc]
})

display(metrics_df)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,0.9825,0.873341,0.857758,0.864869,0.998481,0.883739


In order to conduct some feature engineering , it is helpful to look at the feature importance of each feature importance. 



In [30]:
# Fit SelectFromModel on the DataFrame, not the array

# # Get feature importances from the fitted XGBoost model
# importances = model.feature_importances_

# # Create a DataFrame with column names
# feat_importances = pd.DataFrame({
#     "feature": X_train_prep_df.columns,
#     "importance": importances
# })

# # Sort by importance descending
# feat_importances = feat_importances.sort_values(by="importance", ascending=False)

# # Display
# print(feat_importances)

model_selection = XGBClassifier(
    objective="multi:softmax",
    num_class=7,
    eval_metric="mlogloss",
    random_state=12347303
)

selector = SelectFromModel(xgb_initial, threshold=0.013, prefit=True)

selected_columns = X_train_prep_df.columns[selector.get_support()]

display(selected_columns
        )
X_train_selected = selector.transform(X_resampled)  # DataFrame
X_test_selected  = selector.transform(X_test_prep)   # DataFrame


model_selection.fit(X_train_selected, y_resampled)
y_pred_encoded = model_selection.predict(X_test_selected)
y_pred_labels = pd.Series(y_pred_encoded).map(reverse_grade_mapping)
y_test_series = y_test.squeeze()
y_test_labels = y_test_series.map(reverse_grade_mapping)

f1_macro = f1_score(y_test_encoded, y_pred_encoded, average="macro")
f1_weighted = f1_score(y_test_encoded, y_pred_encoded, average="weighted")

accuracy = accuracy_score(y_test_encoded, y_pred_encoded)
precision = precision_score(y_test_encoded, y_pred_encoded, average='macro')
recall = recall_score(y_test_encoded, y_pred_encoded, average='macro')
f1 = f1_score(y_test_encoded, y_pred_encoded, average='macro')


y_test_bin = label_binarize(y_test_encoded, classes=list(range(7)))
y_pred_proba = model_selection.predict_proba(X_test_selected)
roc_auc = roc_auc_score(y_test_bin, y_pred_proba, average='macro', multi_class='ovr')
pr_auc = average_precision_score(y_test_bin, y_pred_proba, average='macro')


metrics_df = pd.DataFrame({
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc]
})

display(metrics_df)


Index(['purpose_car', 'purpose_moving', 'addr_state_AL', 'addr_state_CO',
       'addr_state_CT', 'addr_state_IL', 'addr_state_LA', 'addr_state_MA',
       'addr_state_WI', 'addr_state_WV', 'pymnt_plan', 'int_rate',
       'issue_d_year'],
      dtype='object')

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,0.983,0.925802,0.947962,0.935874,0.999676,0.98109


We tried the median feature selection, which did not increase the F1 score significantly. By using the average threshhold we got around an f1 score of 0.91, when optimising the treshhold around the mean we found a relative optimal at 0.94 with threshhold 0.013.
We have broken down the feature importance to the following features: purpose_car', 'purpose_house', 'purpose_moving', 'addr_state_AL',
       'addr_state_CO', 'addr_state_CT', 'addr_state_IL', 'addr_state_LA',
       'addr_state_MA', 'addr_state_WI', 'addr_state_WV', 'int_rate',
       'chargeoff_within_12_mths', 'issue_d_year.

       

Now lets try to optomise the hyperparameters. 

In [126]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": [100, 300, 500],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(objective="multi:softmax", num_class=7, random_state=42),
    param_grid=param_grid,
    scoring="f1_micro",
    cv=3,
    verbose=1
)
grid_search.fit(X_train_selected, y_resampled)
print(grid_search.best_params_)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


KeyboardInterrupt: 

Now lets test these parameters to see if it actually improved. 

In [31]:
xgb_after_cv = XGBClassifier(
    objective="multi:softmax", 
    num_class=7,                
    max_depth=7,                
    learning_rate=0.1,          
    n_estimators=100,           
    subsample=1,                
    colsample_bytree=1,         
    random_state=42,           
    eval_metric="mlogloss" 
)


xgb_after_cv.fit(X_train_selected, y_resampled)
y_pred_encoded = xgb_after_cv.predict(X_test_selected)
y_pred_labels = pd.Series(y_pred_encoded).map(reverse_grade_mapping)
y_test_series = y_test.squeeze()
y_test_labels = y_test_series.map(reverse_grade_mapping)

f1_macro = f1_score(y_test_encoded, y_pred_encoded, average="macro")
f1_weighted = f1_score(y_test_encoded, y_pred_encoded, average="weighted")

accuracy = accuracy_score(y_test_encoded, y_pred_encoded)
precision = precision_score(y_test_encoded, y_pred_encoded, average='macro')
recall = recall_score(y_test_encoded, y_pred_encoded, average='macro')
f1 = f1_score(y_test_encoded, y_pred_encoded, average='macro')


y_test_bin = label_binarize(y_test_encoded, classes=list(range(7)))
y_pred_proba = xgb_after_cv.predict_proba(X_test_selected)
roc_auc = roc_auc_score(y_test_bin, y_pred_proba, average='macro', multi_class='ovr')
pr_auc = average_precision_score(y_test_bin, y_pred_proba, average='macro')


metrics_df = pd.DataFrame({
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc]
})

display(metrics_df)


,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,0.9785,0.90829,0.937188,0.919384,0.999609,0.978296


It actually makes it worse therefore it does not really change anythng. 

Now lets look at the selected variables and look for skew or normalisation / scaling that needs to be performed. 

In [32]:
import plotly.express as px
import pandas as pd

cols_to_plot = [
    'purpose_car', 'purpose_house', 'purpose_moving', 
    'addr_state_AL', 'addr_state_CO', 'addr_state_CT', 
    'addr_state_IL', 'addr_state_LA', 'addr_state_MA', 
    'addr_state_WI', 'addr_state_WV', 'int_rate', 
    'chargeoff_within_12_mths', 'issue_d_year'
]

for col in cols_to_plot:
    if str(X_train_prep_df[col].dtype) in ['int64','float64']:  
        fig = px.histogram(
            X_train_prep_df, 
            x=col, 
            marginal="box", 
            nbins=20,
            title=f"{col} distribution & outliers"
        )
    else:  # binary / one-hot
        fig = px.bar(
            X_train_prep_df[col].value_counts().reset_index(), 
            x='index', y=col, 
            title=f"{col} count distribution"
        )
    fig.show()


The dataset (other than one hot-variables) doesnt look for that much variables) 

Now we predict the test csv provided with the best performing model.

In [41]:
X_test_final_features = loan_test_final_df
test_ids = loan_test_final_df["ID"]

X_test_prep_final = pipeline.transform(X_test_final_features)
X_test_final_selected = selector.transform(X_test_prep_final)
y_encoded_test_final_pred = model_selection.predict(X_test_final_selected)
y_pred_series_final = pd.Series(y_encoded_test_final_pred.squeeze())
y_pred_mapped_final = y_pred_series_final.map(reverse_grade_mapping)

output_df = pd.DataFrame({
    'id': test_ids,
    'grade': y_pred_mapped_final
})

output_df.to_csv("predictions_final_xgboost.csv", index=False)


### 2.2 SVM

In [ ]:
svm_model = SVC(decision_function_shape="ovr", kernel="rbf", random_state=12347303)
svm_model.fit(X_train_prep, y_train)

y_pred_encoded = svm_model.predict(X_test_prep)
y_pred_labels = pd.Series(y_pred_encoded).map(reverse_grade_mapping)
y_test_labels = y_test.map(reverse_grade_mapping)

# 5️⃣ Evaluate
f1_macro = f1_score(y_test, y_pred_encoded, average="macro")
f1_weighted = f1_score(y_test, y_pred_encoded, average="weighted")

print("F1-score (macro):", f1_macro)
print("F1-score (weighted):", f1_weighted)

### 2.3 RandomForest

## Model Evaluation 